# Loan Approval Prediction

**Problem type:** Supervised learning – binary classification

**Goal:** Predict whether a loan application will be approved based on the
applicant's financial, demographic, and professional characteristics (age,
income, credit score, loan amount, employment type...).

**Pipeline:**
1. Load the data
2. Exploratory data analysis (EDA)
3. Data cleaning
4. Encoding categorical variables and normalization
5. Training models (Logistic Regression, Random Forest, XGBoost)
6. Evaluation (Accuracy, Precision, Recall, F1-score, ROC-AUC)

## 1. Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Data

This project uses the **"Loan Risk Prediction Dataset"** (Kaggle, author:
sohailkhan05), a synthetically generated set of 5,000 loan applications,
built to simulate a realistic financial loan-approval decision process.

Source: https://www.kaggle.com/datasets/sohailkhan05/loan-risk-prediction

Columns:
- `Age` – applicant's age
- `Gender` – gender (Male / Female)
- `City` – city of residence
- `Education` – highest education level (High School, Bachelors, Masters, PhD)
- `Income` – annual income (contains missing values)
- `LoanAmount` – requested loan amount
- `CreditScore` – credit score (~300–850, contains missing values)
- `EmploymentType` – employment type (Salaried, Self-Employed, Unemployed)
- `YearsExperience` – years of work experience
- `LoanApproved` – target variable (1 = approved, 0 = rejected)

The dataset intentionally contains missing values, mild class imbalance,
and noise in the target variable, to realistically exercise a complete ML
pipeline.

In [ ]:
df = pd.read_csv("loan_risk_prediction_dataset.csv")
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Missing values per column
df.isna().sum()

In [ ]:
# Target variable distribution (class imbalance)
plt.figure(figsize=(5, 4))
sns.countplot(x="LoanApproved", data=df, hue="LoanApproved", palette="Set2", legend=False)
plt.title("Target variable distribution (LoanApproved)")
plt.xlabel("LoanApproved (0 = Rejected, 1 = Approved)")
plt.ylabel("Number of applications")
plt.tight_layout()
plt.show()

In [ ]:
print(df["LoanApproved"].value_counts(normalize=True))

In [ ]:
# Numerical features vs. the target variable
num_cols = ["Age", "Income", "LoanAmount", "CreditScore", "YearsExperience"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(x="LoanApproved", y=col, data=df, hue="LoanApproved",
                palette="Set2", legend=False, ax=axes[i])
    axes[i].set_title(col)
axes[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of numerical features
plt.figure(figsize=(7, 5))
sns.heatmap(df[num_cols + ["LoanApproved"]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Categorical variables vs. approval rate
cat_cols = ["Gender", "Education", "City", "EmploymentType"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    rate = df.groupby(col)["LoanApproved"].mean().sort_values(ascending=False)
    rate.plot(kind="bar", ax=axes[i], color="#4C72B0")
    axes[i].set_title(f"Approval rate by: {col}")
    axes[i].set_ylabel("Share approved")
    axes[i].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 4. Data Cleaning

Issues found in the data:
- Missing values in the `Income` and `CreditScore` columns
- Negative values in `Income` and `LoanAmount` (noise, physically
  impossible) which are treated as invalid and replaced with NaN before
  imputation

Missing/invalid numerical values will be filled with the median inside the
Pipeline (imputation step), which avoids information leakage from the test
set (fitting is done only on the training data).

In [ ]:
# Negative values in Income and LoanAmount are treated as invalid (noise) -> NaN
for col in ["Income", "LoanAmount"]:
    n_negative = (df[col] < 0).sum()
    print(f"{col}: {n_negative} negative values")
    df.loc[df[col] < 0, col] = np.nan

In [ ]:
# Duplicates
print("Number of duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df.shape

## 5. Encoding Categorical Variables, Normalization, and Train/Test Split

Numerical columns: median imputation + standardization (`StandardScaler`).
Categorical columns: most-frequent-value imputation + `OneHotEncoder`.

All transformations are packaged into a `ColumnTransformer`, which is then
used inside a `Pipeline` together with the model, preventing information
leakage from the test set into the training phase.

In [ ]:
X = df.drop(columns=["LoanApproved"])
y = df["LoanApproved"]

numeric_features = ["Age", "Income", "LoanAmount", "CreditScore", "YearsExperience"]
categorical_features = ["Gender", "Education", "City", "EmploymentType"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train.shape, X_test.shape

## 6. Training the Models

Three classification models are trained:
- **Logistic Regression** – a simple, interpretable linear baseline
- **Random Forest** – an ensemble of decision trees, robust to
  non-linearities
- **XGBoost** – gradient boosting, typically the best performer on
  tabular data

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.1,
        eval_metric="logloss", random_state=RANDOM_STATE
    ),
}

pipelines = {}
for name, clf in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe
    print(f"{name}: trained")

## 7. Model Evaluation

Models are compared using: Accuracy, Precision, Recall, F1-score, and
ROC-AUC.

In [ ]:
results = []
roc_curves = {}

for name, pipe in pipelines.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_curves[name] = (fpr, tpr)

results_df = pd.DataFrame(results).set_index("Model").round(4)
results_df

In [ ]:
results_df.plot(kind="bar", figsize=(10, 5))
plt.title("Model comparison across metrics")
plt.ylabel("Metric value")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves
plt.figure(figsize=(6, 5))
for name, (fpr, tpr) in roc_curves.items():
    auc = results_df.loc[name, "ROC-AUC"]
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for the best model (by F1-score)
best_model_name = results_df["F1-score"].idxmax()
best_pipe = pipelines[best_model_name]
y_pred_best = best_pipe.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Rejected", "Approved"])
disp.plot(cmap="Blues", values_format="d")
plt.title(f"Confusion matrix – {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
print(f"Best model by F1-score: {best_model_name}")

## 8. Feature Importance

Feature importance is computed using permutation importance on the best
model, on the test set.

In [ ]:
result = permutation_importance(
    best_pipe, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)

feature_names = X_test.columns
importance = pd.Series(result.importances_mean, index=feature_names).sort_values()

plt.figure(figsize=(7, 5))
importance.plot.barh()
plt.title(f"Feature importance (permutation importance) – {best_model_name}")
plt.xlabel("Drop in Accuracy score")
plt.tight_layout()
plt.show()

## 9. Conclusion

The models successfully learn the relationship between loan approval and
the applicant's financial and professional characteristics. As expected,
`CreditScore`, `Income`, and `EmploymentType` have the strongest influence
on the decision, consistent with real-world credit risk assessment
principles.

XGBoost and Random Forest generally outperform Logistic Regression because
of their ability to capture non-linear relationships between features,
while Logistic Regression remains the simplest and most interpretable
model.

Because of the mild class imbalance (~23% approved applications) and the
noise injected into the target variable, Accuracy alone is not a sufficient
metric – Precision, Recall, F1-score, and ROC-AUC are tracked as well.

Possible directions for further improvement:
- hyperparameter tuning (grid/random search) for all models
- class-balancing techniques (SMOTE, class_weight)
- additional engineered features (e.g. LoanAmount/Income ratio)
- probability calibration for more reliable decision-making